# SGLang Explainer — 2K Colab Render

This notebook renders the Remotion project at **2560 × 1440 (2K QHD), 30 fps** and saves the final **MP4 + render log + manifest** to **Google Drive**.

**Before anything else:** the next cell asks for permission to access your Google Drive and verifies write access. No file is written to Drive until you approve it.

**Runtime advice:** use **Runtime → Change runtime type → High RAM**. A GPU is not required; Remotion renders with Chrome on CPU. A 10-minute 2K render is heavy and can take **1–4 hours** depending on the Colab CPU. Keep the tab open.

The notebook always pulls the latest `main` from:

`https://github.com/lroe/sglang`

Output expected:

- `2560 × 1440`
- H.264 / AAC
- ~9:55 runtime
- roughly 1–2 GB depending on CRF



## 1) Google Drive access (asked first)

The notebook will not start cloning or rendering until Drive access has been granted and a write test has passed.



In [ ]:
#@title Request Google Drive access and choose the output folder
print("Google Drive access is required before this notebook writes anything.")
print("A Google permission dialog will open after you type YES.\n")

consent = input("Type YES to request Google Drive access: ").strip().upper()
if consent != "YES":
    raise SystemExit("Drive access was not approved. Stopping.")

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DEFAULT_OUT = Path("/content/drive/MyDrive/SGLang_Renders")
raw_folder = input(f"Drive output folder [{DEFAULT_OUT}]: ").strip().strip('"').strip("'")
OUT_DIR = Path(raw_folder) if raw_folder else DEFAULT_OUT

if "/content/drive" not in str(OUT_DIR):
    print("⚠️ This folder is not under /content/drive. The notebook will still use it, but it will not be a Drive-backed path.")

OUT_DIR.mkdir(parents=True, exist_ok=True)
probe = OUT_DIR / ".sglang_write_test"
probe.write_text("ok", encoding="utf-8")
assert probe.read_text(encoding="utf-8") == "ok"
probe.unlink()

print(f"\n✅ Drive access verified.")
print(f"📁 Output folder: {OUT_DIR}")




## 2) Pull the latest GitHub source

This clones or updates `lroe/sglang` and prints the exact commit being rendered.



In [ ]:
#@title Clone or update GitHub repo
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/lroe/sglang.git"
REPO_DIR = Path("/content/sglang")

if (REPO_DIR / ".git").exists():
    print("Repo already exists — updating ...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
else:
    print("Cloning ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print(f"\n✅ Repo ready: {REPO_DIR}")
print(f"🔖 Commit: {COMMIT}")




## 3) Install render dependencies

Installs the Chromium runtime libraries Remotion needs, then runs `npm ci` and downloads Remotion’s headless browser.



In [ ]:
#@title System libraries + Node dependencies
import shutil
import subprocess
import sys

# Node 18+ is enough for Remotion 4. Install Node 20 only if Colab is older.
node_version = subprocess.check_output(["node", "--version"], text=True).strip()
node_major = int(node_version.lstrip("v").split(".")[0])
print("Node:", node_version)
if node_major < 18:
    print("Installing Node 20 ...")
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash -", shell=True, check=True)
    subprocess.run("apt-get install -y -qq nodejs", shell=True, check=True)
    print("Node:", subprocess.check_output(["node", "--version"], text=True).strip())

# Libraries needed by Remotion's headless Chromium on a bare Colab image.
apt = (
    "ffmpeg libnss3 libatk-bridge2.0-0 libgtk-3-0 libgbm1 "
    "libxkbcommon0 libxcomposite1 libxdamage1 libxrandr2 libxfixes3 "
    "libcups2 libdrm2 libpango-1.0-0 libcairo2"
)
subprocess.run("apt-get update -qq", shell=True, check=True)
subprocess.run(f"DEBIAN_FRONTEND=noninteractive apt-get install -y -qq {apt}", shell=True, check=True)
# Ubuntu renamed libasound2 in newer images.
subprocess.run(
    "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2 "
    "|| DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2t64",
    shell=True,
    check=True,
)

VIDEO_DIR = REPO_DIR / "video"
print("\nInstalling npm packages ...")
subprocess.run(["npm", "ci", "--no-audit", "--no-fund"], cwd=VIDEO_DIR, check=True)

print("\nDownloading Remotion headless browser ...")
subprocess.run(["npx", "remotion", "browser", "ensure"], cwd=VIDEO_DIR, check=True)

print("\n✅ Dependencies ready.")
print("ffmpeg:", subprocess.check_output(["ffmpeg", "-version"], text=True).splitlines()[0])

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
    print(f"RAM: {ram_gb:.1f} GB")
    if ram_gb < 12:
        print("⚠️ Use a High-RAM runtime if the render runs out of memory.")
except Exception:
    pass




## 4) Optional 3-second smoke test (recommended)

Skip this on a full run if you have already verified the 2K pipeline. It renders frames 0–89 (3 seconds at 30 fps) to `/content/sglang-render/test-2k.mp4` and checks that the output is actually **2560 × 1440**.



In [ ]:
#@title Optional: render 3 seconds at 2K
import os
import subprocess
from pathlib import Path

LOCAL_DIR = Path("/content/sglang-render")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
SMOKE_OUT = LOCAL_DIR / "test-2k.mp4"

env = os.environ.copy()
env["FRAME_RANGE"] = "0-89"

print("Rendering 3-second 2K smoke test ...")
proc = subprocess.Popen(
    ["node", "render-2k.mjs", str(SMOKE_OUT)],
    cwd=VIDEO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Smoke test failed with exit code {rc}")

probe = subprocess.check_output(
    ["ffprobe", "-v", "error", "-select_streams", "v:0",
     "-show_entries", "stream=width,height", "-of", "default=nw=1",
     str(SMOKE_OUT)],
    text=True,
)
print("\nSmoke test output:\n" + probe)
if "width=2560" not in probe or "height=1440" not in probe:
    raise RuntimeError("Smoke test was not 2560x1440 — check render-2k.mjs")
print("✅ Smoke test passed.")




## 5) Render the full video in 2K

This streams the exact frame counts, percentage, render FPS, ETA, and stitching stage in real time. The output first goes to Colab’s local disk, then is copied to Google Drive in the next cell (so Drive I/O cannot corrupt an in-progress render).



In [ ]:
#@title Full 2K render with real-time progress
import os
import subprocess
import time
from pathlib import Path

LOCAL_OUT = Path("/content/sglang-render/sglang-explainer-2k.mp4")
LOG_PATH = Path("/content/sglang-render/render-2k.log")
LOCAL_OUT.parent.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
# Leave CONCURRENCY unset to let render-2k.mjs choose based on CPU count.
# env["CONCURRENCY"] = "4"

started = time.time()
print(f"Rendering full 2K video -> {LOCAL_OUT}")
print(f"Log -> {LOG_PATH}\n")

proc = subprocess.Popen(
    ["node", "render-2k.mjs", str(LOCAL_OUT)],
    cwd=VIDEO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

with LOG_PATH.open("w", encoding="utf-8") as log:
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Render failed with exit code {rc}. See {LOG_PATH}")

elapsed = time.time() - started
print(f"\n✅ Render finished in {elapsed/60:.1f} minutes")
print(f"   {LOCAL_OUT}")




## 6) Verify and copy to Google Drive

Copies the finished MP4, the render log, and a JSON manifest to the Drive folder you approved earlier. The copy prints its own progress too.



In [ ]:
#@title Verify output + copy to Drive
import hashlib
import json
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

def copy_with_progress(src: Path, dst: Path, label: str):
    total = src.stat().st_size
    done = 0
    last = -1
    dst.parent.mkdir(parents=True, exist_ok=True)
    with src.open("rb") as fsrc, dst.open("wb") as fdst:
        while True:
            chunk = fsrc.read(8 * 1024 * 1024)
            if not chunk:
                break
            fdst.write(chunk)
            done += len(chunk)
            pct = int(done * 100 / max(1, total))
            if pct >= last + 5 or done == total:
                print(f"{label}: {pct}%  ({done/1e9:.2f}/{total/1e9:.2f} GB)", flush=True)
                last = pct
    shutil.copystat(src, dst)

probe = subprocess.check_output(
    ["ffprobe", "-v", "error", "-select_streams", "v:0",
     "-show_entries", "stream=width,height,codec_name,duration",
     "-of", "json", str(LOCAL_OUT)],
    text=True,
)
info = json.loads(probe)
stream = info["streams"][0]
width, height = stream["width"], stream["height"]
print(f"Resolution: {width}x{height}")
print(f"Codec: {stream.get('codec_name')}")
print(f"Duration: {stream.get('duration')} s")
if (width, height) != (2560, 1440):
    raise RuntimeError(f"Unexpected output resolution: {width}x{height}")

final_mp4 = OUT_DIR / "sglang-explainer-2k.mp4"
final_log = OUT_DIR / "render-2k.log"
print(f"\nCopying video to Drive -> {final_mp4}")
copy_with_progress(LOCAL_OUT, final_mp4, "video")
copy_with_progress(LOG_PATH, final_log, "log")

manifest = {
    "source_repo": REPO_URL,
    "commit": COMMIT,
    "rendered_at_utc": datetime.now(timezone.utc).isoformat(),
    "resolution": f"{width}x{height}",
    "fps": 30,
    "codec": stream.get("codec_name"),
    "duration_seconds": stream.get("duration"),
    "file": str(final_mp4),
    "log": str(final_log),
}
manifest_path = OUT_DIR / "render-manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"\n✅ Saved to Drive:")
print(f"   {final_mp4}")
print(f"   {final_log}")
print(f"   {manifest_path}")




## Troubleshooting

- **Drive mount fails:** re-run cell 1 and approve the Google permission dialog.
- **Chrome/lib errors:** re-run cell 3; the apt list covers the standard Remotion headless dependencies.
- **Out of memory:** Runtime → Change runtime type → High RAM, then re-run from cell 3.
- **Render is slow:** that is expected at 2560×1440 on a Colab CPU. Concurrency is chosen from `os.cpus()`; set `CONCURRENCY` in the full-render cell if you want to override it.
- **Resume after a disconnect:** Colab does not resume a partial Remotion render. Re-run cells 4–7; the final file is copied to Drive only after a successful complete render.

